# import library

In [0]:
from pyspark.sql.functions import col


# bronze layer

### 1_aisles table

In [0]:
import dlt
@dlt.table(name ="bronze_aisales" , comment = "bronze table for aisles")
def bronze_aisales():
  return spark.sql("""
    select * from cloud_files("/Volumes/workspace/my_project/raw_date/aisales" , "csv" ,
            map(
                "header" ,"true",
                "cloudFiles.inferColumnTypes", "true",
                "cloudFiles.includeExistingFiles", "true", 
                "cloudFiles.schemaEvolutionMode", "addNewColumns"
             
            )
    )        
""")   

### 2_depart table

In [0]:

@dlt.table(name ="bronze_departments" , comment = "bronze table for departments")
def bronze_departments():
  return spark.sql("""
    select * from cloud_files("/Volumes/workspace/my_project/raw_date/depart/" , "csv" ,
            map(
                "header" ,"true",
                "cloudFiles.inferColumnTypes", "true",
                "cloudFiles.includeExistingFiles", "true", 
                "cloudFiles.schemaEvolutionMode", "addNewColumns"
             
            )
    )        
""") 

### 3_order-products-prior table


In [0]:
@dlt.table(name ="bronze_order_prod_prior" , comment = "bronze table for all order products")
def bronze_order_prod_prior():
  return spark.sql("""
    select * from cloud_files("/Volumes/workspace/my_project/raw_date/order_prod_prior/" , "parquet" ,
            map(
            
                "cloudFiles.inferColumnTypes", "true",
                "cloudFiles.includeExistingFiles", "true", 
                "cloudFiles.schemaEvolutionMode", "addNewColumns"
             
            )
    )        
""") 

### 4_order-products-train table



In [0]:
@dlt.table(name ="bronze_order_products_train" , comment = "bronze table for order products")
def bronze_order_products_train():
  return spark.sql("""
    select * from cloud_files("/Volumes/workspace/my_project/raw_date/order_prod_train/" , "csv" ,
            map(
                "header" ,"true",
                "cloudFiles.inferColumnTypes", "true",
                "cloudFiles.includeExistingFiles", "true", 
                "cloudFiles.schemaEvolutionMode", "addNewColumns"
             
            )
    )        
""") 

### 5_order table

In [0]:
@dlt.table(name ="bronze_orders" , comment = "bronze table for orders")
def bronze_orders():
  return spark.sql("""
    select * from cloud_files("/Volumes/workspace/my_project/raw_date/orders/" , "csv" ,
            map(
                "header" ,"true",
                "cloudFiles.inferColumnTypes", "true",
                "cloudFiles.includeExistingFiles", "true", 
                "cloudFiles.schemaEvolutionMode", "addNewColumns"
             
            )
    )        
""") 

### 6_products table

In [0]:
@dlt.table(name ="bronze_products" , comment = "bronze table for products")
def bronze_products():
  return spark.sql("""
    select * from cloud_files("/Volumes/workspace/my_project/raw_date/proudctes/" , "csv" ,
            map(
                "header" ,"true",
                "cloudFiles.inferColumnTypes", "true",
                "cloudFiles.includeExistingFiles", "true", 
                "cloudFiles.schemaEvolutionMode", "addNewColumns"
             
            )
    )        
""") 

# silver layer-----------------------------

#### 1_aisles table

In [0]:
@dlt.table(name ="silver_aisales" , comment = "silver table for aisales")
def silver_aisales():
  return spark.sql("""
    select 
      try_cast(aisle_id AS INTEGER),
      try_cast(aisle AS INTEGER) AS aisle_name
         

     FROM bronze_aisales
    
  """)

#### 2_departments table

In [0]:
@dlt.table(name ="silver_departments" , comment = "silver table for aisales")
def silver_departments():
  return spark.sql("""
    select 
      department_id,
      department
         
    FROM bronze_departments
    
    """)

#### 3_order_products_prior table

In [0]:
@dlt.table(name ="silver_order_prod_prior" , comment = "silver table for order_products_prior")
def silver_order_prod_prior():
  return spark.sql("""
    select 
      try_cast(order_id AS INTEGER) AS order_id,
      try_cast(product_id AS INTEGER) AS product_id ,
      cast(add_to_cart_order AS INTEGER) AS add_to_cart_order,
      CASE
        when cast(reordered AS INTEGER) = 1 then TRUE
        when cast(reordered AS INTEGER) = 0 then FALSE
        ELSE NULL
      END AS is_reordered
         
    FROM bronze_order_prod_prior
    
    """)


#### 4_order_products_train table

In [0]:
@dlt.table(name ="silver_order_prod_train" , comment = "silver table for order_products_train")
def silver_order_prod_train():
  return spark.sql("""
    select 
      try_cast(order_id AS INTEGER) AS order_id,
      try_cast(product_id AS INTEGER) AS product_id ,
      cast(add_to_cart_order AS INTEGER) AS add_to_cart_order,
      CASE
        when cast(reordered AS INTEGER) = 1 then TRUE
        when cast(reordered AS INTEGER) = 0 then FALSE
        ELSE NULL
      END AS is_reordered
         
    FROM bronze_order_products_train
    
    """)

#### 5_orders table

In [0]:
@dlt.table(name ="silver_orders" , comment = "silver table for orders")
def silver_orders():
  return spark.sql("""
    select 
      try_cast(order_id AS INTEGER) AS order_id,
      try_cast(user_id AS INTEGER) AS user_id,
      CASE 
        WHEN LOWER(TRIM(eval_set)) IN ('prior', 'train', 'test') THEN LOWER(TRIM(eval_set))
        ELSE NULL 
       END AS evaluation_set,
      CASE
        when try_cast(order_number AS INTEGER) > 0 then order_number 
        ELSE NULL
      END AS order_number,
      CASE 
        WHEN order_dow BETWEEN 0 AND 6 THEN order_dow
        ELSE NULL 
      END AS order_day_of_week,
      
      CASE 
        WHEN order_hour_of_day BETWEEN 0 AND 23 THEN order_hour_of_day
        ELSE NULL 
      END AS order_hour_of_day,

      CASE 
        WHEN order_number = 1 THEN NULL 
        ELSE days_since_prior_order  
      END AS days_since_prior_order


         
    FROM bronze_orders
    
    """)

#### 6_prodctes table

In [0]:
@dlt.table(name ="silver_products" , comment = "silver table for products")
def silver_products():
  return spark.sql("""
    select 
      try_cast(product_id AS INTEGER) AS product_id,
      COALESCE(NULLIF(TRIM(product_name), ''), 'N/A') AS product_name,
      try_cast(aisle_id AS INTEGER) AS aisle_id,
      try_cast(department_id AS INTEGER) AS department_id

    FROM bronze_products
    
    """) 

# gold layer------------------------------------

### 1_ Product dimension table

In [0]:


@dlt.table(
    name="gold_dim_products",
    comment="Product dimension table enriched with aisle and department names for Gold Layer",
    
)
def gold_dim_products():
    return spark.sql("""
        SELECT 
            p.product_id,
            p.product_name,
            p.aisle_id,
            a.aisle_name,
            p.department_id,
            d.department
        FROM silver_products p
        LEFT JOIN silver_aisales a 
            ON p.aisle_id = a.aisle_id
        LEFT JOIN silver_departments d 
            ON p.department_id = d.department_id
    """)

### orders fact table

In [0]:


@dlt.table(
    name="gold_fact_orders",
    comment="Order Fact table combining prior and train order products using Subquery",
    
)
def gold_fact_orders():
    return spark.sql("""
        SELECT 
            op.order_id,
            o.user_id,
            op.product_id,
            op.add_to_cart_order,
            op.is_reordered,
            o.order_number,
            o.order_day_of_week,
            o.order_hour_of_day,
            o.days_since_prior_order
        FROM (
            SELECT order_id, product_id, add_to_cart_order, is_reordered
            FROM silver_order_prod_prior
            
            UNION ALL
            
            SELECT order_id, product_id, add_to_cart_order, is_reordered
            FROM silver_order_prod_train
        ) op
        INNER JOIN silver_orders o 
            ON op.order_id = o.order_id
    """)